# 02 — Model 1: Donut Encoder (Swin Transformer)

> Notebook ini adalah **hasil run asli di Kaggle** (GPU T4) lengkap dengan output-nya. Isi teknis di bawah ditulis dalam bahasa Inggris saat eksperimen.

**Peran dalam workflow:** model gambar pertama (baseline). Menggunakan encoder Donut yang dipretrain pada dokumen, dengan resolusi tinggi 1024x768 agar detail teks pada scan masih terlihat.

| | |
|---|---|
| Input | Data kompetisi |
| Output | `best.pt`, `val_predictions.csv`, `val_probs.npy`, `test_probs.npy`, `submission.csv` |
| Durasi | ~7,5 jam (3 epoch x 145 menit) |
| Hasil | Holdout **0,8919** · Public **0,88122** |

Kode yang sama dalam bentuk modul ber-docstring: `src/models.py` (`DonutClassifier`), `src/engine.py`, `src/data.py`, `configs/donut.yaml`.

# RVL-CDIP Document Classification — Donut Encoder Baseline

**Before running (Kaggle settings panel):**
- Accelerator: **GPU T4** (single GPU is used)
- Internet: **ON** (downloads `naver-clova-ix/donut-base` from Hugging Face)

**How to run:**
1. Set `QUICK_RUN = True` and run all cells. This is a ~10-minute smoke test on a small subset to check paths, memory and the full loop.
2. Set `QUICK_RUN = False`, then **Save Version → Save & Run All (Commit)**, so the long run continues in the background.

**Pipeline:** locate data → audit (drop only corrupt / fully blank train pages) → stratified 90/10 holdout →
fine-tune the Donut Swin encoder + linear head → evaluate on the holdout → predict test → `submission.csv`.

Outputs in `/kaggle/working`: `best.pt`, `submission.csv`, `val_predictions.csv`, `test_probs.npy`, `audit.csv`.

⚠️ Do **not** use `naver-clova-ix/donut-base-finetuned-rvlcdip`. It was trained on full RVL-CDIP, which almost certainly includes these test images (leakage).

## 1. Config

In [1]:
import os, gc, math, random, time, json
from pathlib import Path
from multiprocessing import Pool

import numpy as np
import pandas as pd
from PIL import Image, ImageStat
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import VisionEncoderDecoderModel, get_cosine_schedule_with_warmup

QUICK_RUN = False            # True = smoke test on a subset; False = full training run

# Resolution: RVL-CDIP pages are ~1000 px on the long side, so 1024x768 keeps roughly native detail.
# Going higher only upsamples. If you hit OOM or the run is too slow, use 768x576.
IMG_H, IMG_W = 1024, 768

BATCH_SIZE      = 8
GRAD_ACCUM      = 2         # effective batch = 16
EPOCHS          = 3
LR_ENCODER      = 3e-5
LR_HEAD         = 1e-3
WEIGHT_DECAY    = 0.05
WARMUP_RATIO    = 0.05
LABEL_SMOOTHING = 0.1       # cheap robustness against the ~8% label noise in RVL-CDIP
VAL_FRAC        = 0.10
SEED            = 42
NUM_WORKERS     = os.cpu_count() or 2

DROP_BLANK = True           # drop only train pages with essentially zero content
BLANK_STD  = 1.0            # pixel std (0-255) on a thumbnail; very strict on purpose

MODEL_NAME = "naver-clova-ix/donut-base"   # pretrained only, NOT the rvlcdip-finetuned checkpoint
OUT = Path("/kaggle/working")
RESUME = True               # resume from last.pt if it exists in /kaggle/working

if QUICK_RUN:
    EPOCHS = 1

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_everything(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
print("CPU workers:", NUM_WORKERS)

Device: cuda Tesla T4
CPU workers: 4


## 2. Locate data and resolve image paths
`train.csv` lists paths like `train/invoice/x.tif`, but the actual folders are `train/train/invoice/x.tif`. The resolver handles both layouts.

In [2]:
cands = sorted(Path("/kaggle/input").rglob("train.csv"))
assert cands, "train.csv not found under /kaggle/input - did you add the competition data?"
ROOT = cands[0].parent
print("Data root:", ROOT)

train_df = pd.read_csv(ROOT / "train.csv")
test_df  = pd.read_csv(ROOT / "test.csv")
sample_sub = pd.read_csv(ROOT / "sample_submission.csv")

def resolve(rel):
    rel = str(rel)
    p1 = ROOT / rel
    if p1.exists():
        return str(p1)
    p2 = ROOT / Path(rel).parts[0] / rel      # e.g. ROOT/train/train/invoice/x.tif
    if p2.exists():
        return str(p2)
    return None

train_df["path"] = [resolve(f) for f in tqdm(train_df["filename"], desc="resolve train")]
test_df["path"]  = [resolve(f) for f in tqdm(test_df["filename"],  desc="resolve test")]
print("Missing train files:", train_df["path"].isna().sum(), "| missing test files:", test_df["path"].isna().sum())
assert test_df["path"].notna().all(), "Some test files could not be located - check the folder layout."
train_df = train_df[train_df["path"].notna()].reset_index(drop=True)

# label map, checked for consistency between class_name and label
cls_map = train_df.groupby("label")["class_name"].agg(lambda s: s.unique().tolist())
assert all(len(v) == 1 for v in cls_map), "class_name <-> label is not one-to-one"
CLASSES = [cls_map[i][0] for i in sorted(cls_map.index)]
NUM_CLASSES = len(CLASSES)
print(NUM_CLASSES, "classes:", CLASSES)
print(train_df["class_name"].value_counts().to_string())

Data root: /kaggle/input/competitions/document-image-classification-sd234504


resolve train:   0%|          | 0/32003 [00:00<?, ?it/s]

resolve test:   0%|          | 0/7994 [00:00<?, ?it/s]

Missing train files: 0 | missing test files: 0
16 classes: ['advertisement', 'budget', 'email', 'file_folder', 'form', 'handwritten', 'invoice', 'letter', 'memo', 'news_article', 'presentation', 'questionnaire', 'resume', 'scientific_publication', 'scientific_report', 'specification']
class_name
scientific_publication    2056
resume                    2029
handwritten               2026
file_folder               2022
email                     2013
advertisement             2012
form                      2005
budget                    2004
scientific_report         1999
memo                      1994
presentation              1992
invoice                   1982
specification             1978
letter                    1972
news_article              1971
questionnaire             1948


## 3. Audit: corrupt and fully blank pages
Hard-but-valid pages (faint scans, near-empty folders, rotated handwriting) are **kept**. Only unreadable files and pages with essentially zero pixel variation are dropped from train. Test pages are never dropped; a corrupt test page gets a white placeholder image.

In [3]:
def audit_one(path):
    try:
        with Image.open(path) as im:
            im.seek(0)
            w, h = im.size
            g = im.convert("L")
            g.thumbnail((256, 256))
            st = ImageStat.Stat(g)
            return (path, True, w, h, st.mean[0], st.stddev[0], "")
    except Exception as e:
        return (path, False, 0, 0, np.nan, np.nan, str(e)[:200])

audit_paths = list(train_df["path"]) + list(test_df["path"])
if QUICK_RUN:
    audit_paths = audit_paths[:2000]
with Pool(NUM_WORKERS) as pool:
    rows = list(tqdm(pool.imap(audit_one, audit_paths, chunksize=64), total=len(audit_paths), desc="audit"))
audit = pd.DataFrame(rows, columns=["path", "ok", "w", "h", "mean", "std", "err"])
audit.to_csv(OUT / "audit.csv", index=False)

print("Unreadable files:", (~audit["ok"]).sum())
print("Size (w x h) median:", int(audit.loc[audit.ok, "w"].median()), "x", int(audit.loc[audit.ok, "h"].median()))
print("Blank pages (std <", BLANK_STD, "):", (audit["std"] < BLANK_STD).sum())

train_df = train_df.merge(audit[["path", "ok", "std"]], on="path", how="left")
bad = (train_df["ok"] == False) | (DROP_BLANK & (train_df["std"] < BLANK_STD))
print("\nTrain rows to drop:", int(bad.sum()))
if bad.any():
    print(train_df.loc[bad, "class_name"].value_counts().to_string())   # check these classes before trusting the drop
train_df = train_df[~bad].reset_index(drop=True)

bad_test = test_df["path"].isin(audit.loc[~audit.ok, "path"])
print("Unreadable test files (will use placeholder):", int(bad_test.sum()))

audit:   0%|          | 0/39997 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:982: UserWarning: Corrupt EXIF data.  Expecting to read 2 bytes but only got 0. 
  warnings.warn(str(msg))


Unreadable files: 1
Size (w x h) median: 759 x 1000
Blank pages (std < 1.0 ): 0

Train rows to drop: 1
class_name
scientific_publication    1
Unreadable test files (will use placeholder): 0


## 4. Holdout split
This 10% holdout keeps the **original labels** and is never cleaned. Any later VLM relabeling must beat this baseline on this same split.

In [4]:
tr_df, va_df = train_test_split(train_df, test_size=VAL_FRAC, stratify=train_df["label"], random_state=SEED)
tr_df, va_df = tr_df.reset_index(drop=True), va_df.reset_index(drop=True)
te_df = test_df.copy()

if QUICK_RUN:
    tr_df = tr_df.groupby("label", group_keys=False).apply(lambda d: d.sample(min(len(d), 40), random_state=SEED)).reset_index(drop=True)
    va_df = va_df.groupby("label", group_keys=False).apply(lambda d: d.sample(min(len(d), 10), random_state=SEED)).reset_index(drop=True)
    te_df = te_df.head(64).copy()

print(f"train {len(tr_df)} | val {len(va_df)} | test {len(te_df)}")

train 28801 | val 3201 | test 7994


## 5. Dataset and augmentation
There's no denoising, binarizing or cropping, because train and test share the same scan conditions. Augmentation is light and scan-like. There's no horizontal flip, since that would mirror the text.

In [5]:
train_aug = T.Compose([
    T.RandomApply([T.RandomAffine(degrees=2, translate=(0.02, 0.02), scale=(0.95, 1.05), fill=255)], p=0.5),
    T.RandomApply([T.ColorJitter(brightness=0.2, contrast=0.2)], p=0.3),
    T.RandomApply([T.GaussianBlur(kernel_size=3)], p=0.1),
])

def load_page(path):
    try:
        with Image.open(path) as im:
            im.seek(0)
            im = im.convert("L")                 # 1-bit / RGB -> grayscale
    except Exception:
        im = Image.new("L", (IMG_W, IMG_H), 255)
    # Pillow's resize is antialiased, so thin strokes in 1-bit scans survive as gray instead of vanishing
    return im.resize((IMG_W, IMG_H), Image.Resampling.BILINEAR)

class DocDataset(Dataset):
    def __init__(self, df, train=False):
        self.paths = df["path"].tolist()
        self.labels = df["label"].tolist() if "label" in df else [-1] * len(df)
        self.train = train
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        im = load_page(self.paths[i])
        if self.train:
            im = train_aug(im)
        x = TF.to_tensor(im)                     # 1 x H x W in [0, 1]
        x = ((x - 0.5) / 0.5).repeat(3, 1, 1)    # Donut normalization, 3 channels
        return x, self.labels[i]

def make_loader(df, train):
    return DataLoader(DocDataset(df, train), batch_size=BATCH_SIZE, shuffle=train,
                      num_workers=NUM_WORKERS, pin_memory=True, drop_last=train,
                      persistent_workers=NUM_WORKERS > 0)

train_loader = make_loader(tr_df, True)
val_loader   = make_loader(va_df, False)
test_loader  = make_loader(te_df, False)

## 6. Model: Donut Swin encoder + classification head
The Donut text decoder is discarded. Only the document-pretrained Swin encoder is kept, followed by mean pooling and a linear head.

In [6]:
class DonutClassifier(nn.Module):
    def __init__(self, name, n_classes):
        super().__init__()
        ved = VisionEncoderDecoderModel.from_pretrained(name)
        self.encoder = ved.encoder
        del ved; gc.collect()
        hid = self.encoder.config.hidden_size
        self.head = nn.Sequential(nn.LayerNorm(hid), nn.Dropout(0.1), nn.Linear(hid, n_classes))
    def forward(self, x):
        h = self.encoder(pixel_values=x).last_hidden_state   # B x tokens x hid
        return self.head(h.mean(dim=1))

model = DonutClassifier(MODEL_NAME, NUM_CLASSES)
try:
    model.encoder.gradient_checkpointing_enable()   # trades ~30% speed for much lower memory
    print("Gradient checkpointing: on")
except Exception as e:
    print("Gradient checkpointing unavailable:", e)
model.to(DEVICE)
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

config.json:   0%|          | 0.00/4.74k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  809MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  809MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/484 [00:00<?, ?it/s]

Gradient checkpointing: on
Params: 74.2M


## 7. Optimizer, scheduler, loss

In [7]:
def param_groups(model):
    groups = {"enc_decay": [], "enc_nodecay": [], "head": []}
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if n.startswith("head"):
            groups["head"].append(p)
        elif p.ndim == 1 or n.endswith(".bias") or "norm" in n.lower():
            groups["enc_nodecay"].append(p)
        else:
            groups["enc_decay"].append(p)
    return [
        {"params": groups["enc_decay"],   "lr": LR_ENCODER, "weight_decay": WEIGHT_DECAY},
        {"params": groups["enc_nodecay"], "lr": LR_ENCODER, "weight_decay": 0.0},
        {"params": groups["head"],        "lr": LR_HEAD,    "weight_decay": WEIGHT_DECAY},
    ]

optimizer = torch.optim.AdamW(param_groups(model))
steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_steps = steps_per_epoch * EPOCHS
scheduler = get_cosine_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_steps), total_steps)
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE == "cuda")
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

start_epoch, best_acc = 0, 0.0
last_ckpt = OUT / "last.pt"
if RESUME and last_ckpt.exists() and not QUICK_RUN:
    ck = torch.load(last_ckpt, map_location="cpu")
    model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optimizer"])
    scheduler.load_state_dict(ck["scheduler"]); scaler.load_state_dict(ck["scaler"])
    start_epoch, best_acc = ck["epoch"] + 1, ck["best_acc"]
    print(f"Resumed from epoch {start_epoch}, best acc {best_acc:.4f}")

/tmp/ipykernel_22/3629288047.py:22: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=DEVICE == "cuda")


## 8. Train and validate

In [8]:
@torch.no_grad()
def predict(loader):
    model.eval()
    probs = []
    for x, _ in tqdm(loader, desc="predict", leave=False):
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
            logits = model(x.to(DEVICE, non_blocking=True))
        probs.append(logits.float().softmax(-1).cpu().numpy())
    return np.concatenate(probs)

for epoch in range(start_epoch, EPOCHS):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    t0, run_loss = time.time(), 0.0
    pbar = tqdm(train_loader, desc=f"epoch {epoch+1}/{EPOCHS}")
    for step, (x, y) in enumerate(pbar):
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
            loss = criterion(model(x), y) / GRAD_ACCUM
        scaler.scale(loss).backward()
        if (step + 1) % GRAD_ACCUM == 0 or step + 1 == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
        run_loss += loss.item() * GRAD_ACCUM
        pbar.set_postfix(loss=f"{run_loss/(step+1):.4f}")

    val_probs = predict(val_loader)
    val_pred = val_probs.argmax(1)
    acc = accuracy_score(va_df["label"], val_pred)
    mf1 = f1_score(va_df["label"], val_pred, average="macro")
    print(f"epoch {epoch+1}: train_loss {run_loss/len(train_loader):.4f} | val_acc {acc:.4f} | "
          f"val_macroF1 {mf1:.4f} | {(time.time()-t0)/60:.1f} min")

    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), OUT / "best.pt")
        print("  saved best.pt")
    if not QUICK_RUN:
        torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                    "epoch": epoch, "best_acc": best_acc}, last_ckpt)

print("Best val acc:", best_acc)

epoch 1/3:   0%|          | 0/3600 [00:00<?, ?it/s]

/tmp/ipykernel_22/4229640811.py:26: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


predict:   0%|          | 0/401 [00:00<?, ?it/s]

epoch 1: train_loss 1.4007 | val_acc 0.8341 | val_macroF1 0.8321 | 145.0 min
  saved best.pt


epoch 2/3:   0%|          | 0/3600 [00:00<?, ?it/s]

predict:   0%|          | 0/401 [00:00<?, ?it/s]

epoch 2: train_loss 0.9734 | val_acc 0.8882 | val_macroF1 0.8880 | 144.9 min
  saved best.pt


epoch 3/3:   0%|          | 0/3600 [00:00<?, ?it/s]

predict:   0%|          | 0/401 [00:00<?, ?it/s]

epoch 3: train_loss 0.8540 | val_acc 0.8919 | val_macroF1 0.8915 | 145.0 min
  saved best.pt
Best val acc: 0.8919087785067167


## 9. Holdout error analysis
The top confused pairs show where the errors actually are. Use this to decide whether text, more resolution or label cleaning is the next step.

In [9]:
model.load_state_dict(torch.load(OUT / "best.pt", map_location=DEVICE))
val_probs = predict(val_loader)
val_pred = val_probs.argmax(1)
y_true = va_df["label"].values

print(classification_report(y_true, val_pred, labels=list(range(NUM_CLASSES)),
                            target_names=CLASSES, digits=3, zero_division=0))

cm = confusion_matrix(y_true, val_pred, labels=list(range(NUM_CLASSES)))
pairs = [(cm[i, j], CLASSES[i], CLASSES[j]) for i in range(NUM_CLASSES) for j in range(NUM_CLASSES) if i != j and cm[i, j] > 0]
print("Top confusions (count, true -> predicted):")
for c, a, b in sorted(pairs, reverse=True)[:12]:
    print(f"  {c:4d}  {a} -> {b}")

val_out = va_df[["id", "path", "label", "class_name"]].copy()
val_out["pred"] = val_pred
val_out["pred_name"] = [CLASSES[p] for p in val_pred]
val_out["conf"] = val_probs.max(1)
val_out.to_csv(OUT / "val_predictions.csv", index=False)
np.save(OUT / "val_probs.npy", val_probs)

predict:   0%|          | 0/401 [00:00<?, ?it/s]

                        precision    recall  f1-score   support

         advertisement      0.887     0.896     0.891       201
                budget      0.846     0.876     0.861       201
                 email      0.995     0.990     0.993       201
           file_folder      0.873     0.955     0.913       202
                  form      0.781     0.711     0.745       201
           handwritten      0.923     0.951     0.937       203
               invoice      0.894     0.894     0.894       198
                letter      0.875     0.888     0.882       197
                  memo      0.944     0.930     0.937       199
          news_article      0.923     0.858     0.889       197
          presentation      0.808     0.869     0.838       199
         questionnaire      0.889     0.867     0.878       195
                resume      0.980     0.970     0.975       203
scientific_publication      0.926     0.917     0.922       206
     scientific_report      0.789     0

## 10. Predict test and write the submission
`test_probs.npy` is saved for ensembling later (e.g., with a DiT model).

In [10]:
test_probs = predict(test_loader)
np.save(OUT / "test_probs.npy", test_probs)
pred_df = pd.DataFrame({"id": te_df["id"].values, "label": test_probs.argmax(1)})

if QUICK_RUN:
    print("QUICK_RUN: skipping submission.csv. Pipeline OK. Set QUICK_RUN = False and commit.")
    print(pred_df.head())
else:
    sub = sample_sub[["id"]].merge(pred_df, on="id", how="left")
    assert sub["label"].notna().all() and len(sub) == len(sample_sub), "Submission is missing ids"
    sub["label"] = sub["label"].astype(int)
    sub.to_csv(OUT / "submission.csv", index=False)
    print(sub.head())
    print("Prediction distribution:\n", sub["label"].map(dict(enumerate(CLASSES))).value_counts().to_string())

predict:   0%|          | 0/1000 [00:00<?, ?it/s]

           id  label
0  test_00001      0
1  test_00002      0
2  test_00003      1
3  test_00004      6
4  test_00005     13
Prediction distribution:
 label
file_folder               534
presentation              529
form                      520
handwritten               517
letter                    513
scientific_report         508
advertisement             506
email                     505
budget                    504
resume                    496
specification             488
scientific_publication    486
invoice                   482
news_article              477
memo                      468
questionnaire             461
